# Quaks News Analyst — ETL Test

This notebook simulates the `quaks_insights_news` Airflow DAG locally for testing and debugging.

It exercises the full HTTP flow that the DAG performs:
1. Authenticate with the backend using service account credentials
2. Create integration + language model + agent
3. Send `BATCH_ETL` message to trigger the full news pipeline
4. Index the resulting report into Elasticsearch (`quaks_insights-news_usa`)

**Prerequisites:** Set the following environment variables in your `.env` file:
- `QUAKS_API_URL` — Backend endpoint (e.g. `https://quaks.ai` or `http://localhost:18000`)
- `QUAKS_SERVICE_ACCOUNT_USERNAME` — Service account username
- `QUAKS_SERVICE_ACCOUNT_SECRET` — Service account password
- `QUAKS_INTEGRATION_API_KEY` — LLM provider API key (e.g. xAI)
- `ELASTICSEARCH_URL` — Elasticsearch endpoint
- `ELASTICSEARCH_API_KEY` — Elasticsearch API key

In [3]:
import os
import hashlib
import json
from uuid import uuid4
from datetime import datetime

import requests
from IPython.display import HTML, display
from dotenv import load_dotenv

os.chdir("..")
load_dotenv()

QUAKS_API_URL = os.getenv("QUAKS_API_URL", "https://quaks.ai")
USERNAME = os.getenv("QUAKS_SERVICE_ACCOUNT_USERNAME")
PASSWORD = os.getenv("QUAKS_SERVICE_ACCOUNT_SECRET")
INTEGRATION_TYPE = os.getenv("QUAKS_INTEGRATION_TYPE", "xai_api_v1")
INTEGRATION_API_KEY = os.getenv("QUAKS_INTEGRATION_API_KEY")
LANGUAGE_MODEL_TAG = os.getenv("QUAKS_LANGUAGE_MODEL_TAG", "grok-4-1-fast-non-reasoning")
ES_URL = os.getenv("ELASTICSEARCH_URL")
ES_API_KEY = os.getenv("ELASTICSEARCH_API_KEY")

INTEGRATION_ENDPOINTS = {
    "xai_api_v1": "https://api.x.ai/v1/",
    "openai_api_v1": "https://api.openai.com/v1/",
    "anthropic_api_v1": "https://api.anthropic.com",
}

print(f"API URL: {QUAKS_API_URL}")
print(f"Integration: {INTEGRATION_TYPE}")
print(f"Model: {LANGUAGE_MODEL_TAG}")

API URL: https://quaks.ai
Integration: xai_api_v1
Model: grok-4-1-fast-non-reasoning


### Step 1: Authenticate

In [4]:
login_response = requests.post(
    f"{QUAKS_API_URL}/auth/login",
    json={"username": USERNAME, "password": PASSWORD},
)
login_response.raise_for_status()
access_token = login_response.json()["access_token"]
headers = {"Authorization": f"Bearer {access_token}"}
print("Authentication successful.")

Authentication successful.


### Step 2: Create Integration + Language Model + Agent

In [5]:
# Create integration
api_endpoint = INTEGRATION_ENDPOINTS.get(INTEGRATION_TYPE)
integration_response = requests.post(
    f"{QUAKS_API_URL}/integrations/create",
    json={
        "integration_type": INTEGRATION_TYPE,
        "api_endpoint": api_endpoint,
        "api_key": INTEGRATION_API_KEY,
    },
    headers=headers,
)
integration_response.raise_for_status()
integration_id = integration_response.json()["id"]
print(f"Integration created: {integration_id}")

# Create language model
llm_response = requests.post(
    f"{QUAKS_API_URL}/llms/create",
    json={
        "integration_id": integration_id,
        "language_model_tag": LANGUAGE_MODEL_TAG,
    },
    headers=headers,
)
llm_response.raise_for_status()
language_model_id = llm_response.json()["id"]
print(f"Language model created: {language_model_id}")

# Create agent
agent_name = f"insights_news_{uuid4()}"
agent_response = requests.post(
    f"{QUAKS_API_URL}/agents/create",
    json={
        "agent_name": agent_name,
        "agent_type": "quaks_news_analyst",
        "language_model_id": language_model_id,
    },
    headers=headers,
)
agent_response.raise_for_status()
agent_id = agent_response.json()["id"]
print(f"Agent created: {agent_id}")

Integration created: 9a3c2283-d868-42a6-a751-ce286a89dfdc
Language model created: a97f40e8-c67a-4b38-82f7-317d59640f7d
Agent created: 40298841-607f-4200-b4b2-49a9f41c1695


### Step 3: Send BATCH_ETL Message

In [8]:
print("Sending BATCH_ETL message (this may take a few minutes)...")
message_response = requests.post(
    f"{QUAKS_API_URL}/messages/post",
    json={
        "agent_id": agent_id,
        "message_role": "human",
        "message_content": "BATCH_ETL",
    },
    headers=headers,
    timeout=600,
)
message_response.raise_for_status()
result = message_response.json()

response_data = result.get("response_data", {})
report_html = response_data.get("report_html", result.get("message_content", ""))
executive_summary = response_data.get("executive_summary", "")

print(f"Executive summary: {executive_summary[:200]}")
print(f"Report length: {len(report_html)} chars")

Sending BATCH_ETL message (this may take a few minutes)...
Executive summary: Mixed markets amid U.S.-Iran tensions, with oil surging despite record IEA reserve release, AI stocks rallying on Oracle's blowout earnings, and crypto consolidating as Trump taps strategic reserves.
Report length: 6695 chars


### Step 4: Preview Report

In [9]:
display(HTML(report_html))

### Step 5: Index into Elasticsearch

In [10]:
today = datetime.now().strftime("%Y-%m-%d")
doc_id = hashlib.md5(f"insights_news_{today}_{agent_id}".encode("utf-8")).hexdigest()
index_name = "quaks_insights-news_usa"

doc = {
    "date_reference": today,
    "text_executive_summary": executive_summary,
    "text_report_html": report_html,
}

bulk_body = json.dumps({"index": {"_index": index_name, "_id": doc_id}}) + "\n" + json.dumps(doc) + "\n"

print(f"Indexing into {index_name} (doc_id: {doc_id})...")
es_response = requests.post(
    f"{ES_URL}/_bulk",
    headers={
        "Authorization": f"ApiKey {ES_API_KEY}",
        "Content-Type": "application/x-ndjson",
    },
    data=bulk_body.encode("utf-8"),
)
es_response.raise_for_status()
print(f"Indexing result: {es_response.json()}")

Indexing into quaks_insights-news_usa (doc_id: 586125ce41ab84a1c29ad8497199ce2f)...
Indexing result: {'errors': False, 'took': 0, 'items': [{'index': {'_index': 'quaks_insights-news_usa', '_id': '586125ce41ab84a1c29ad8497199ce2f', '_version': 1, 'result': 'created', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 0, '_primary_term': 1, 'status': 201}}]}
